In [ ]:
#import libraries

%pip install pandas numpy
import pandas as pd
import numpy as np
import pyarrow as pa
import os

for ext_name in ['pandas.interval', 'pandas.period']:
    try:
        pa.unregister_extension_type(ext_name)
    except KeyError:
        pass

# Define file paths based on the required repository structure
TAXI_DATA_PATH = '../data/raw/tlc/yellow_tripdata_2023-01.parquet'
ZONE_LOOKUP_PATH = '../data/raw/lookup/taxi_zone_lookup.csv'
OUTPUT_CURATED_PATH = '../data/curated/part1_taxi_curated.parquet'


# Load the raw datasets
df = pd.read_parquet(TAXI_DATA_PATH, engine ='fastparquet')
df_zone = pd.read_csv(ZONE_LOOKUP_PATH)



Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Merge for Pickup Location details
df = df.merge(
    df_zone[['LocationID', 'Borough', 'Zone']], 
    left_on='PULocationID', 
    right_on='LocationID', 
    how='left'
).rename(columns={'Borough': 'PU_borough', 'Zone': 'PU_zone'}).drop(columns=['LocationID'])

# Merge for Dropoff Location details
df = df.merge(
    df_zone[['LocationID', 'Borough', 'Zone']], 
    left_on='DOLocationID', 
    right_on='LocationID', 
    how='left'
).rename(columns={'Borough': 'DO_borough', 'Zone': 'DO_zone'}).drop(columns=['LocationID'])

In [ ]:
# --- B. Time fields ---
# Ensure datetime format
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df['pickup_date'] = df['tpep_pickup_datetime'].dt.date
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['weekday'] = df['tpep_pickup_datetime'].dt.day_name().str[:3] # Mon-Sun
df['week_of_year'] = df['tpep_pickup_datetime'].dt.isocalendar().week

# --- D. Engineered fields ---
# trip_duration_min
df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60.0

# speed_mph (handle zero duration safely)
df['speed_mph'] = np.where(
    df['trip_duration_min'] > 0, 
    df['trip_distance'] / (df['trip_duration_min'] / 60.0), 
    0.0
)

# fare_per_mile (handle zero distance safely)
df['fare_per_mile'] = np.where(
    df['trip_distance'] > 0, 
    df['fare_amount'] / df['trip_distance'], 
    0.0
)

# tip_rate (handle zero total amount safely)
df['tip_rate'] = np.where(
    df['total_amount'] > 0, 
    df['tip_amount'] / df['total_amount'], 
    0.0
)

# distance_bucket with bins: [0,1), [1,3), [3,5), [5,10), [10,+)
bins = [0, 1, 3, 5, 10, np.inf]
labels = ['[0,1)', '[1,3)', '[3,5)', '[5,10)', '[10,+)']
df['distance_bucket'] = pd.cut(df['trip_distance'], bins=bins, labels=labels, right=False)

In [50]:
# Initialize tracker for the before/after table
cleaning_tracker = []
initial_count = len(df)

# Rule 1: Remove rows where pickup or dropoff timestamp is missing
df_clean = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])
count_r1 = len(df_clean)
cleaning_tracker.append({
    'Rule': '1. Missing pickup/dropoff timestamp', 
    'Rows Before': initial_count, 
    'Rows After': count_r1, 
    'Removed': initial_count - count_r1
})

# Rule 2: Remove rows where trip_duration_min <= 0
df_clean = df_clean[df_clean['trip_duration_min'] > 0]
count_r2 = len(df_clean)
cleaning_tracker.append({
    'Rule': '2. trip_duration_min <= 0', 
    'Rows Before': count_r1, 
    'Rows After': count_r2, 
    'Removed': count_r1 - count_r2
})

# Rule 3: Remove rows where trip_distance <= 0
df_clean = df_clean[df_clean['trip_distance'] > 0]
count_r3 = len(df_clean)
cleaning_tracker.append({
    'Rule': '3. trip_distance <= 0', 
    'Rows Before': count_r2, 
    'Rows After': count_r3, 
    'Removed': count_r2 - count_r3
})

# Rule 4: Remove rows where passenger_count is missing or <= 0
df_clean = df_clean[(df_clean['passenger_count'].notnull()) & (df_clean['passenger_count'] > 0)]
count_r4 = len(df_clean)
cleaning_tracker.append({
    'Rule': '4. passenger_count missing or <= 0', 
    'Rows Before': count_r3, 
    'Rows After': count_r4, 
    'Removed': count_r3 - count_r4
})

# Rule 5: Remove rows where total_amount <= 0
df_clean = df_clean[df_clean['total_amount'] > 0]
count_r5 = len(df_clean)
cleaning_tracker.append({
    'Rule': '5. total_amount <= 0', 
    'Rows Before': count_r4, 
    'Rows After': count_r5, 
    'Removed': count_r4 - count_r5
})

# Rule 6: Remove duplicates using the specified key
dup_key = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'total_amount']
df_clean = df_clean.drop_duplicates(subset=dup_key)
count_r6 = len(df_clean)
cleaning_tracker.append({
    'Rule': '6. Duplicates (by defined key)', 
    'Rows Before': count_r5, 
    'Rows After': count_r6, 
    'Removed': count_r5 - count_r6
})

# Print the before/after table
tracker_df = pd.DataFrame(cleaning_tracker)
print("Cleaning Rules Summary Table:")
display(tracker_df) # Use display() for a nicer formatted table in Jupyter

Cleaning Rules Summary Table:


,Rule,Rows Before,Rows After,Removed
0,1. Missing pickup/dropoff timestamp,3066766,3066766,0
1,2. trip_duration_min <= 0,3066766,3065645,1121
2,3. trip_distance <= 0,3065645,3020831,44814
3,4. passenger_count missing or <= 0,3020831,2906607,114224
4,5. total_amount <= 0,2906607,2884397,22210
5,6. Duplicates (by defined key),2884397,2884396,1


In [ ]:
# Define the final list of required columns
final_columns = [
    # Geography
    'PULocationID', 'DOLocationID', 'PU_borough', 'PU_zone', 'DO_borough', 'DO_zone',
    # Time
    'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'pickup_date', 'pickup_hour', 'weekday', 'week_of_year',
    # Core numeric
    'passenger_count', 'trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'payment_type',
    # Engineered
    'trip_duration_min', 'speed_mph', 'fare_per_mile', 'tip_rate', 'distance_bucket'
]

# Select final columns
df_final = df_clean[final_columns].copy()

# Ensure the output directory exists
os.makedirs(os.path.dirname(OUTPUT_CURATED_PATH), exist_ok=True)

#for col in df_final.columns:
    #if str(df_final[col].dtype).startswith("period"):
        #df_final[col] = df_final[col].astype(str)

# Save to parquet
df_final.to_parquet(OUTPUT_CURATED_PATH, index=False, engine='pyarrow')
print(f"Curated dataset successfully saved to: {OUTPUT_CURATED_PATH}")